# Black-Litterman Portfolio Optimization - Comprehensive Example

This notebook demonstrates the complete workflow of the Black-Litterman model:

1. Data preparation and market equilibrium
2. Incorporating investor views
3. Calculating posterior distributions
4. Portfolio optimization
5. Visualization and analysis

## Mathematical Background

The Black-Litterman model combines market equilibrium (CAPM) with investor views using Bayesian inference:

$$E[R] = [(\tau\Sigma)^{-1} + P'\Omega^{-1}P]^{-1} [(\tau\Sigma)^{-1}\Pi + P'\Omega^{-1}Q]$$

where:
- $\Pi$ = Prior expected returns (market equilibrium)
- $\Sigma$ = Covariance matrix
- $P$ = Picking matrix (defines views)
- $Q$ = View expected returns
- $\Omega$ = Uncertainty in views
- $\tau$ = Scaling factor (typically 0.025)

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from black_litterman import BlackLittermanModel
from data_utils import prepare_bl_inputs
from visualization import (
    plot_efficient_frontier,
    plot_portfolio_weights,
    plot_return_attribution,
    plot_correlation_matrix,
    create_portfolio_dashboard
)

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.precision', 4)

## 1. Data Preparation

We'll use a diversified portfolio of major assets across different sectors.

In [ ]:
# Define our universe of assets
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'BAC', 'XOM', 'CVX', 'JNJ', 'PFE']
asset_names = [
    'Apple', 'Microsoft', 'Google', 'Amazon', 
    'JPMorgan', 'Bank of America', 'ExxonMobil', 'Chevron',
    'Johnson & Johnson', 'Pfizer'
]

# Prepare all Black-Litterman inputs
data = prepare_bl_inputs(
    tickers=tickers,
    period='3y',  # 3 years of historical data
    cov_method='ledoit_wolf',  # Shrinkage estimator
    risk_free_rate=0.03  # 3% risk-free rate
)

print(f"\nAssets: {', '.join(asset_names)}")
print(f"Data shape: {data['returns'].shape}")

## 2. Initialize Black-Litterman Model

The model starts with market equilibrium derived from market capitalizations.

In [ ]:
# Initialize the Black-Litterman model
bl_model = BlackLittermanModel(
    cov_matrix=data['cov_matrix'],
    market_caps=data['market_caps'],
    risk_aversion=data['risk_aversion'],
    tau=0.025,
    risk_free_rate=data['risk_free_rate']
)

# Display initial equilibrium
print("\n" + "="*60)
print("MARKET EQUILIBRIUM (Prior)")
print("="*60)

equilibrium_df = pd.DataFrame({
    'Asset': asset_names,
    'Market Weight': bl_model.market_weights,
    'Equilibrium Return': bl_model.pi
})

print(equilibrium_df.to_string(index=False))
print(f"\nRisk Aversion Coefficient: {bl_model.risk_aversion:.2f}")

In [ ]:
# Visualize correlation structure
fig = plot_correlation_matrix(
    bl_model.cov_matrix,
    asset_names=asset_names,
    title="Asset Correlation Matrix"
)
plt.show()

## 3. Incorporating Investor Views

Now we add our subjective views about future returns. The Black-Litterman framework supports:

- **Absolute views**: "Asset X will return Y%"
- **Relative views**: "Asset X will outperform Asset Y by Z%"
- **Custom views**: Any linear combination of assets

Each view has a confidence level (0-1), where higher confidence gives the view more weight.

In [ ]:
# Example views based on market outlook

# View 1: Tech sector (AAPL) will outperform with high confidence
bl_model.add_absolute_view(
    asset_idx=0,  # Apple
    expected_return=0.12,  # 12% expected return
    confidence=0.8  # High confidence
)

# View 2: Microsoft will outperform Google by 3%
bl_model.add_relative_view(
    asset_idx1=1,  # Microsoft
    asset_idx2=2,  # Google
    outperformance=0.03,  # 3% outperformance
    confidence=0.6  # Moderate confidence
)

# View 3: Energy sector (XOM) will have moderate returns
bl_model.add_absolute_view(
    asset_idx=6,  # ExxonMobil
    expected_return=0.08,  # 8% expected return
    confidence=0.5  # Medium confidence
)

# View 4: Healthcare (JNJ) will outperform Financials (JPM)
bl_model.add_relative_view(
    asset_idx1=8,  # Johnson & Johnson
    asset_idx2=4,  # JPMorgan
    outperformance=0.02,  # 2% outperformance
    confidence=0.7  # Good confidence
)

print("\n" + "="*60)
print("INVESTOR VIEWS ADDED")
print("="*60)
print(f"Number of views: {len(bl_model.Q)}")
print(f"\nView returns: {bl_model.Q}")
print(f"\nPicking matrix (P):")
print(pd.DataFrame(bl_model.P, columns=asset_names))

## 4. Calculate Posterior Distribution

The Black-Litterman model combines our views with market equilibrium using Bayesian inference.

In [ ]:
# Calculate posterior expected returns and covariance
posterior_returns, posterior_cov = bl_model.calculate_posterior()

# Display summary
summary = bl_model.summary(asset_names=asset_names)

print("\n" + "="*60)
print("BLACK-LITTERMAN POSTERIOR RETURNS")
print("="*60)
print(summary.to_string())

# Calculate statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Average prior return: {bl_model.pi.mean()*100:.2f}%")
print(f"Average posterior return: {posterior_returns.mean()*100:.2f}%")
print(f"Max view impact: {summary['View_Impact'].max()*100:.2f}%")
print(f"Min view impact: {summary['View_Impact'].min()*100:.2f}%")

In [ ]:
# Visualize return attribution
fig = plot_return_attribution(
    bl_model,
    asset_names=asset_names
)
plt.show()

## 5. Portfolio Optimization

We can now optimize our portfolio using different strategies.

In [ ]:
# Optimize for maximum Sharpe ratio
print("\n" + "="*60)
print("PORTFOLIO OPTIMIZATION")
print("="*60)

weights_max_sharpe = bl_model.optimize_portfolio(
    optimization_type='max_sharpe',
    allow_short=False,
    l2_gamma=0.1  # Small regularization for diversification
)

# Calculate portfolio metrics
metrics = bl_model.calculate_portfolio_metrics(weights_max_sharpe)

print("\nMaximum Sharpe Ratio Portfolio:")
print(f"  Expected Return: {metrics['expected_return']*100:.2f}%")
print(f"  Volatility: {metrics['volatility']*100:.2f}%")
print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.3f}")
print(f"  Effective Number of Assets: {metrics['effective_assets']:.2f}")

# Display weights
weights_df = pd.DataFrame({
    'Asset': asset_names,
    'Weight': weights_max_sharpe
}).sort_values('Weight', ascending=False)

print("\nPortfolio Weights:")
print(weights_df.to_string(index=False))

In [ ]:
# Visualize portfolio weights
fig = plot_portfolio_weights(
    weights_max_sharpe,
    asset_names=asset_names,
    title="Optimal Portfolio Weights (Max Sharpe Ratio)"
)
plt.show()

## 6. Efficient Frontier Analysis

Calculate and visualize the efficient frontier.

In [ ]:
# Calculate efficient frontier
print("\nCalculating efficient frontier...")

frontier_returns, frontier_vols, frontier_weights = bl_model.calculate_efficient_frontier(
    n_points=100,
    allow_short=False
)

print(f"Frontier points calculated: {len(frontier_returns)}")
print(f"Return range: {frontier_returns.min()*100:.2f}% to {frontier_returns.max()*100:.2f}%")
print(f"Volatility range: {frontier_vols.min()*100:.2f}% to {frontier_vols.max()*100:.2f}%")

In [ ]:
# Plot efficient frontier
fig = plot_efficient_frontier(
    frontier_returns,
    frontier_vols,
    optimal_weights=weights_max_sharpe,
    posterior_returns=posterior_returns,
    posterior_cov=posterior_cov,
    asset_names=asset_names,
    risk_free_rate=bl_model.risk_free_rate,
    title="Black-Litterman Efficient Frontier",
    show_assets=True
)
plt.show()

## 7. Alternative Optimization Strategies

In [ ]:
# Minimum volatility portfolio
weights_min_vol = bl_model.optimize_portfolio(
    optimization_type='min_volatility',
    allow_short=False
)

metrics_min_vol = bl_model.calculate_portfolio_metrics(weights_min_vol)

# Target return portfolio
target_return = 0.10  # 10% target
weights_target = bl_model.optimize_portfolio(
    optimization_type='target_return',
    target_return=target_return,
    allow_short=False
)

metrics_target = bl_model.calculate_portfolio_metrics(weights_target)

# Comparison table
comparison = pd.DataFrame({
    'Strategy': ['Max Sharpe', 'Min Volatility', f'Target {target_return*100}%'],
    'Return': [
        f"{metrics['expected_return']*100:.2f}%",
        f"{metrics_min_vol['expected_return']*100:.2f}%",
        f"{metrics_target['expected_return']*100:.2f}%"
    ],
    'Volatility': [
        f"{metrics['volatility']*100:.2f}%",
        f"{metrics_min_vol['volatility']*100:.2f}%",
        f"{metrics_target['volatility']*100:.2f}%"
    ],
    'Sharpe Ratio': [
        f"{metrics['sharpe_ratio']:.3f}",
        f"{metrics_min_vol['sharpe_ratio']:.3f}",
        f"{metrics_target['sharpe_ratio']:.3f}"
    ]
})

print("\n" + "="*60)
print("PORTFOLIO STRATEGY COMPARISON")
print("="*60)
print(comparison.to_string(index=False))

## 8. Comprehensive Dashboard

Generate a complete portfolio analysis dashboard.

In [ ]:
# Create comprehensive dashboard
fig = create_portfolio_dashboard(
    bl_model,
    weights_max_sharpe,
    asset_names=asset_names
)
plt.show()

## 9. Sensitivity Analysis

Analyze how portfolio composition changes with different confidence levels.

In [ ]:
# Test different confidence levels for first view
confidence_levels = [0.1, 0.3, 0.5, 0.7, 0.9]
sensitivity_results = []

for conf in confidence_levels:
    # Create new model
    bl_temp = BlackLittermanModel(
        cov_matrix=data['cov_matrix'],
        market_caps=data['market_caps'],
        risk_aversion=data['risk_aversion'],
        tau=0.025,
        risk_free_rate=data['risk_free_rate']
    )
    
    # Add view with varying confidence
    bl_temp.add_absolute_view(
        asset_idx=0,  # Apple
        expected_return=0.12,
        confidence=conf
    )
    
    bl_temp.calculate_posterior()
    weights_temp = bl_temp.optimize_portfolio(optimization_type='max_sharpe', allow_short=False)
    
    sensitivity_results.append({
        'Confidence': conf,
        'Apple Weight': weights_temp[0],
        'Portfolio Return': (bl_temp.posterior_returns @ weights_temp),
        'Portfolio Vol': np.sqrt(weights_temp @ bl_temp.posterior_cov @ weights_temp)
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

print("\n" + "="*60)
print("SENSITIVITY ANALYSIS: View Confidence Impact")
print("="*60)
print(sensitivity_df.to_string(index=False))

# Plot sensitivity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(sensitivity_df['Confidence'], sensitivity_df['Apple Weight']*100, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('View Confidence', fontweight='bold')
ax1.set_ylabel('Apple Weight (%)', fontweight='bold')
ax1.set_title('Impact of Confidence on Apple Allocation', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(sensitivity_df['Confidence'], sensitivity_df['Portfolio Return']*100, 'o-', linewidth=2, markersize=8, label='Return')
ax2_twin = ax2.twinx()
ax2_twin.plot(sensitivity_df['Confidence'], sensitivity_df['Portfolio Vol']*100, 'o-', linewidth=2, markersize=8, color='orange', label='Volatility')
ax2.set_xlabel('View Confidence', fontweight='bold')
ax2.set_ylabel('Expected Return (%)', fontweight='bold', color='blue')
ax2_twin.set_ylabel('Volatility (%)', fontweight='bold', color='orange')
ax2.set_title('Portfolio Risk-Return vs Confidence', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Export Results

In [ ]:
# Create comprehensive output report
output_report = {
    'Summary': summary,
    'Optimal_Weights': weights_df,
    'Portfolio_Metrics': pd.DataFrame([metrics]),
    'Strategy_Comparison': comparison
}

# Save to Excel (optional)
# with pd.ExcelWriter('bl_portfolio_results.xlsx') as writer:
#     for sheet_name, df in output_report.items():
#         df.to_excel(writer, sheet_name=sheet_name)

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)
print("\nKey Takeaways:")
print(f"1. Optimal portfolio achieves {metrics['sharpe_ratio']:.3f} Sharpe ratio")
print(f"2. Portfolio is diversified across {metrics['effective_assets']:.1f} effective assets")
print(f"3. Views successfully incorporated with impact range: {summary['View_Impact'].min()*100:.2f}% to {summary['View_Impact'].max()*100:.2f}%")
print(f"4. Expected portfolio return: {metrics['expected_return']*100:.2f}% with {metrics['volatility']*100:.2f}% volatility")

## Conclusion

This notebook demonstrated the full capabilities of the Black-Litterman model:

1. **Market Equilibrium**: Started from CAPM-implied returns based on market caps
2. **View Integration**: Incorporated subjective views with confidence levels
3. **Bayesian Combination**: Blended views with equilibrium using mathematical rigor
4. **Optimization**: Generated optimal portfolios under various strategies
5. **Analysis**: Performed sensitivity analysis and comprehensive visualization

The Black-Litterman model provides a principled way to incorporate views while respecting market equilibrium, leading to more stable and diversified portfolios compared to traditional mean-variance optimization.

### Next Steps

- Experiment with different views and confidence levels
- Try different asset universes (bonds, commodities, international stocks)
- Implement dynamic rebalancing strategies
- Backtest portfolio performance
- Add transaction costs and constraints